In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_7.py ──
"""
Shared infrastructure for MLFP04 Exercise 7 — Recommender Systems.

Scenario: Singapore e-commerce marketplace (similar to Shopee/Lazada).
  - 100 users x 50 SKUs from a fictional SG electronics retailer
  - 30% ratings observed (explicit 1-5 stars)
  - Holdout 20% of observed ratings for evaluation
  - Item features = 8 latent product attributes (price tier, category, etc.)

Contains: synthetic rating matrix generation, evaluation metrics (RMSE,
precision@k, MAP), shared constants, and output-dir setup.

Technique-specific code (content-based profile, user/item CF, ALS, hybrid
blending) lives in the per-technique files.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex7_recommenders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS — synthetic SG e-commerce dataset
# ════════════════════════════════════════════════════════════════════════

N_USERS = 100
N_ITEMS = 50
N_LATENT_TRUE = 5
N_ITEM_FEATURES = 8
SPARSITY = 0.30
HOLDOUT_FRAC = 0.20
RNG_SEED = 42

RATING_MIN = 1.0
RATING_MAX = 5.0
RELEVANCE_THRESHOLD = 3.5  # ratings >= 3.5 are "relevant" for ranking metrics


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC RATING MATRIX — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def build_rating_dataset(
    seed: int = RNG_SEED,
) -> dict:
    """Generate the shared synthetic e-commerce rating matrix.

    Returns a dict with:
      R_observed       — (N_USERS, N_ITEMS) with NaN where not rated
      R_train          — R_observed with holdout entries set to NaN
      mask             — boolean observed mask
      train_mask       — boolean training-only mask
      holdout_mask     — boolean holdout mask
      U_true, V_true   — ground-truth latent factors (for subspace recovery)
      item_features    — (N_ITEMS, N_ITEM_FEATURES) content-feature matrix
      user_ids, item_ids — stable string IDs
      ratings_df       — long-format polars DataFrame of observed ratings
    """
    rng = np.random.default_rng(seed=seed)

    U_true = rng.normal(0, 1, size=(N_USERS, N_LATENT_TRUE))
    V_true = rng.normal(0, 1, size=(N_ITEMS, N_LATENT_TRUE))

    R_full = U_true @ V_true.T
    R_full = (R_full - R_full.min()) / (R_full.max() - R_full.min()) * 4 + 1
    R_full += rng.normal(0, 0.3, size=R_full.shape)
    R_full = np.clip(R_full, RATING_MIN, RATING_MAX)

    mask = rng.random(size=(N_USERS, N_ITEMS)) < SPARSITY
    R_observed = np.where(mask, R_full, np.nan)

    holdout_mask = mask & (rng.random(size=(N_USERS, N_ITEMS)) < HOLDOUT_FRAC)
    train_mask = mask & ~holdout_mask
    R_train = np.where(train_mask, R_observed, np.nan)

    item_features = rng.random(size=(N_ITEMS, N_ITEM_FEATURES))

    user_ids = [f"sg_user_{i:03d}" for i in range(N_USERS)]
    item_ids = [f"sku_{j:02d}" for j in range(N_ITEMS)]

    rows = []
    for i in range(N_USERS):
        for j in range(N_ITEMS):
            if mask[i, j]:
                rows.append(
                    {
                        "user_id": user_ids[i],
                        "item_id": item_ids[j],
                        "rating": round(float(R_observed[i, j]), 1),
                        "in_holdout": bool(holdout_mask[i, j]),
                    }
                )
    ratings_df = pl.DataFrame(rows)

    return {
        "R_observed": R_observed,
        "R_train": R_train,
        "mask": mask,
        "train_mask": train_mask,
        "holdout_mask": holdout_mask,
        "U_true": U_true,
        "V_true": V_true,
        "item_features": item_features,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "ratings_df": ratings_df,
        "rng": rng,
    }


# ════════════════════════════════════════════════════════════════════════
# EVALUATION METRICS — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def holdout_rmse(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
) -> tuple[float, float]:
    """Return (RMSE, coverage) on the holdout mask.

    Coverage = fraction of holdout pairs where the method produced a
    non-NaN prediction. Methods that can't predict cold pairs have
    coverage < 1.0.
    """
    errors = []
    n_holdout = int(holdout_mask.sum())
    for i in range(predictions.shape[0]):
        for j in range(predictions.shape[1]):
            if holdout_mask[i, j] and not np.isnan(predictions[i, j]):
                errors.append((R_true[i, j] - predictions[i, j]) ** 2)
    if not errors:
        return float("inf"), 0.0
    rmse = float(np.sqrt(np.mean(errors)))
    coverage = len(errors) / max(n_holdout, 1)
    return rmse, coverage


def precision_at_k(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    k: int = 5,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Precision@k averaged across users.

    For each user, rank holdout items by predicted score, take the top-k,
    and compute what fraction of those have true rating >= threshold.
    """
    precisions = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])
        top = [j for j, _ in scored[:k]]
        if not top:
            continue
        precisions.append(len(set(top) & relevant) / len(top))
    return float(np.mean(precisions)) if precisions else 0.0


def mean_average_precision(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Mean Average Precision across users on the holdout set."""
    aps = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])

        hits = 0
        sum_precision = 0.0
        for rank, (j, _) in enumerate(scored, 1):
            if j in relevant:
                hits += 1
                sum_precision += hits / rank
        if hits > 0:
            aps.append(sum_precision / len(relevant))
    return float(np.mean(aps)) if aps else 0.0


def print_method_scores(
    name: str, preds: np.ndarray, R_true: np.ndarray, holdout_mask: np.ndarray
) -> dict:
    """Print a single-line scorecard and return the metrics dict."""
    rmse, cov = holdout_rmse(preds, R_true, holdout_mask)
    p5 = precision_at_k(preds, R_true, holdout_mask, k=5)
    m = mean_average_precision(preds, R_true, holdout_mask)
    print(
        f"  {name:<22} RMSE={rmse:6.4f}  coverage={cov:6.1%}  "
        f"P@5={p5:6.4f}  MAP={m:6.4f}"
    )
    return {"RMSE": rmse, "Coverage": cov, "P@5": p5, "MAP": m}


# ════════════════════════════════════════════════════════════════════════
# VISUAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def save_html(fig, filename: str) -> Path:
    """Write a plotly fig into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    print(f"  saved: {path}")
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 7.2: User-Based Collaborative Filtering
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compute pairwise user similarity on mean-centred ratings
#   - Borrow preferences from the top-k most similar users
#   - Understand why mean-centring fixes "generous rater" bias
#   - See the failure modes: scale, sparsity, and cold-start users
#
# PREREQUISITES: Exercise 7.1 (content-based baseline)
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — "users who agreed with me before will agree with me again"
#   2. Build — user similarity matrix + top-k CF prediction
#   3. Train — no training; neighbourhoods are looked up at inference
#   4. Visualise — user similarity heatmap + neighbour quality
#   5. Apply — Singapore streaming watchlist expansion
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.express as px


K_NEIGHBOURS = 20



## THEORY — Why user-based CF works


In [ ]:
# If Alice and Bob agreed on 50 past items, Bob's opinion on a new item
# is strong evidence for Alice. Find each user's nearest neighbours in
# rating space and predict by a weighted average of their ratings.
#
# Mean-centring per user removes "generous rater" bias so the similarity
# compares rankings, not absolute scores.



## TASK 2 — BUILD similarity + predictor


Pairwise cosine similarity between users on mean-centred ratings.


Predict ratings using the top-k most similar users.


In [ ]:
def user_similarity_matrix(
    R: np.ndarray, obs_mask: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    n_users = R.shape[0]
    sim = np.zeros((n_users, n_users))

    # TODO: Compute each user's mean rating (only over observed entries).
    # Use np.nanmean on R[u, obs_mask[u]] and default to 0.0 when empty.
    user_means = np.array([____ for u in range(n_users)])

    # TODO: Build a mean-centred copy of R. Subtract user_means[u] from
    # each user's observed entries and zero out the unobserved entries.
    R_centred = R.copy()
    for u in range(n_users):
        ____
    R_centred[~obs_mask] = 0.0

    for u in range(n_users):
        for v in range(u, n_users):
            both = obs_mask[u] & obs_mask[v]
            if not both.any():
                continue
            ru, rv = R_centred[u, both], R_centred[v, both]
            denom = np.linalg.norm(ru) * np.linalg.norm(rv)
            if denom < 1e-10:
                continue
            # TODO: cosine similarity of centred vectors ru, rv
            s = ____
            sim[u, v] = s
            sim[v, u] = s

    return sim, user_means


def user_based_cf_predict(
    R: np.ndarray,
    obs_mask: np.ndarray,
    sim: np.ndarray,
    user_means: np.ndarray,
    k: int = K_NEIGHBOURS,
) -> np.ndarray:
    n_users, n_items = R.shape
    predictions = np.full((n_users, n_items), np.nan)

    for u in range(n_users):
        similarities = sim[u].copy()
        similarities[u] = -np.inf
        # TODO: Pick the top-k indices by similarity. Hint: np.argsort(...)[-k:]
        top_k = ____
        top_k = top_k[similarities[top_k] > 0]
        if len(top_k) == 0:
            continue

        for j in range(n_items):
            rated_neighbours = top_k[obs_mask[top_k, j]]
            if len(rated_neighbours) == 0:
                continue
            weights = sim[u, rated_neighbours]
            denom = np.abs(weights).sum()
            if denom < 1e-10:
                continue
            # TODO: weighted deviation of neighbours' ratings from their means
            weighted_dev = ____
            predictions[u, j] = user_means[u] + weighted_dev / denom

    return np.clip(predictions, 1.0, 5.0)



## TASK 3 — Precompute similarity, then inference is a lookup


In [ ]:
print("\n" + "=" * 70)
print("  User-Based CF on SG E-commerce Ratings")
print("=" * 70)

data = build_rating_dataset()
R_train = data["R_train"]
R_observed = data["R_observed"]
train_mask = data["train_mask"]
holdout_mask = data["holdout_mask"]

user_sim, user_means = user_similarity_matrix(R_train, train_mask)
ubcf_predictions = user_based_cf_predict(
    R_train, train_mask, user_sim, user_means, k=K_NEIGHBOURS
)



### Checkpoint


In [ ]:
ubcf_rmse, ubcf_cov = holdout_rmse(ubcf_predictions, R_observed, holdout_mask)
assert user_sim.shape == (N_USERS, N_USERS), "User similarity must be N x N"
assert np.allclose(user_sim, user_sim.T), "Similarity must be symmetric"
assert ubcf_rmse > 0, "User-CF RMSE should be positive"
print(
    f"\n[ok] Checkpoint passed — User-CF holdout RMSE={ubcf_rmse:.4f}, "
    f"coverage={ubcf_cov:.1%}\n"
)



## TASK 4 — VISUALISE similarity structure


In [ ]:
order = np.argsort(user_means)
sim_sorted = user_sim[np.ix_(order, order)]

fig = px.imshow(
    sim_sorted,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="User-User Similarity Heatmap (sorted by mean rating)",
    labels={"x": "user j", "y": "user i", "color": "cosine sim"},
)
save_html(fig, "02_user_similarity_heatmap.html")

neighbour_counts = [(user_sim[u] > 0).sum() - 1 for u in range(N_USERS)]
print(
    f"Neighbourhood stats: mean={np.mean(neighbour_counts):.1f}, "
    f"min={np.min(neighbour_counts)}, max={np.max(neighbour_counts)}"
)

print_method_scores("User-CF", ubcf_predictions, R_observed, holdout_mask)



## TASK 5 — APPLY: Singapore Streaming Watchlist Expansion


In [ ]:
# SCENARIO: A SG/SEA streaming service (~2M MAU) runs a "because users
# like you enjoyed..." row on its homepage. User-CF captures community
# taste that content features miss.
#
# BUSINESS IMPACT: 18% watch-through lift ~ S$4.3M/year retained
# subscription revenue on a 2M MAU S$12 ARPU platform.



## REFLECTION


[x] Computed pairwise user similarity on mean-centred ratings
  [x] Borrowed preferences from top-k nearest neighbours
  [x] Understood why mean-centring beats raw cosine
  [x] Identified a SEA streaming scenario with S$4M/year in impact

  Next: 03_item_cf.py — flip the direction and scale the algorithm.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

